In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df = pd.read_csv('data.csv', encoding='utf-8')
df.head()

In [ ]:
print(df.isnull().sum())

In [ ]:
missing_val = (df.isnull().sum()/len(df))*100
print(missing_val)

In [ ]:
df = df.drop_duplicates()
print(df.duplicated().sum())

In [ ]:
df["Club"] = df["Club"].fillna("Unknown")
print(df.isnull().sum())


In [ ]:
max_goals = df["Goals"].max()
max_player = df.loc[df["Goals"].idxmax(), "Player Names"]
print(f"Player name: {max_player}, Max goals: {max_goals}")

In [ ]:
# player_name = input("Enter Player Name: ")

# # Search player in dataset
# player = df[df["Player Names"].str.lower() == player_name.lower()]

# # Check if player exists
# if not player.empty:
    
#     # Display required details
#     print("\nPlayer Details")
#     print("-------------------")
    
#     print("Player :", player.iloc[0]["Player Names"])
#     print("Goals  :", player.iloc[0]["Goals"])
#     print("Matches :", player.iloc[0]["Matches_Played"])
#     print("Shots  :", player.iloc[0]["Shots"])
    
# else:
#     print("Player not found in dataset")

In [ ]:
df.dtypes
df.head()

In [ ]:
df.columns = df.columns.str.strip()

In [ ]:
from sklearn.preprocessing import StandardScaler

df.columns = df.columns.str.strip()
col = ['Matches_Played','Substitution','Mins','xG','Shots','OnTarget']
std_obj = StandardScaler()
df[col] = std_obj.fit_transform(df[col])
df

In [ ]:
features = [
    'Matches_Played',
    'Mins',
    'xG Per Avg Match',
    'Shots',
    'OnTarget',
    'Shots Per Avg Match',
    'On Target Per Avg Match'
]

In [ ]:
target = ['Goals']

In [ ]:
X = df[features]
y = df[target]

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,
    y,
    test_size=0.2,
    random_state=42)
print(X_train.shape)
print(X_test.shape)




In [ ]:
from xgboost import XGBRegressor

model = XGBRegressor(
   n_estimators=500,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


# Train model
model = XGBRegressor()
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Evaluation
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print("R2 Score:", r2)
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("Accuracy:", r2 * 100)

In [ ]:
predictions = model.predict(X_test)

comparison = pd.DataFrame({
    'Actual_Goals': y_test.values.ravel(),
    'Predicted_Goals': predictions.ravel()
})

print(comparison.head())



In [ ]:
# Convert to 1D arrays
y_test_flat = np.ravel(y_test)
predictions_flat = np.ravel(y_pred)

plt.figure(figsize=(12,6))

# Actual goals → Red
plt.plot(
    y_test_flat[:50],
    color='red',
    label='Actual Goals',
    linewidth=2
)

# Predicted goals → Blue
plt.plot(
    predictions_flat[:50],
    color='blue',
    label='Predicted Goals',
    linewidth=2
)

plt.xlabel("Players")
plt.ylabel("Goals")

plt.title("Actual Goals vs Predicted Goals(on 50 players)")

plt.legend()

plt.grid(True)

plt.show()

In [ ]:


# Convert arrays to 1D
y_test_flat = np.ravel(y_test)
predictions_flat = np.ravel(y_pred)

# Create figure
plt.figure(figsize=(12,6))

# Actual Goals → Red
sns.lineplot(
    x=range(len(y_test_flat[:])),
    y=y_test_flat[:],
    color='red',
    label='Actual Goals'
)

# Predicted Goals → Blue
sns.lineplot(
    x=range(len(predictions_flat[:])),
    y=predictions_flat[:],
    color='blue',
    label='Predicted Goals'
)

# Labels
plt.xlabel("Players")
plt.ylabel("Goals")

# Title
plt.title("Actual Goals vs Predicted Goals(all players)")

# Legend
plt.legend()

# Grid
plt.grid(True)

# Show graph
plt.show()

In [ ]:
import pickle

# Save trained model
with open("model.pkl", "wb") as file:
    pickle.dump(model, file)